## Full Training Script
---
Run the following code cell after your data loading and device definition. It assumes you already have X_train, y_train, X_val, y_val, X_test, y_test, INPUT_DIM, DEVICE, FINAL_EPOCHS, MODELS_DIR, RESULTS_DIR defined.

In [23]:
import time
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import f1_score, accuracy_score, precision_score, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
# Ruta donde están tus CSVs (ajústala si es diferente)
DATA_DIR = Path("../dataset/processed")   # o donde tengas los archivos
# Rutas específicas de CARGA DE DATOS
MODELS_DIR = Path("models")  
RESULTS_DIR = Path("results")  
RESULTS_DIR.mkdir(exist_ok=True)
FINAL_EPOCHS = 20
DEVICE = torch.device("cuda")
print(f"Device: {DEVICE}")
INPUT_DIM = 13

# Cargar los tres archivos
train_df = pd.read_csv(DATA_DIR / "HTML_train_scaled.csv")
val_df   = pd.read_csv(DATA_DIR / "HTML_val_scaled.csv")
test_df  = pd.read_csv(DATA_DIR / "HTML_test_scaled.csv")

# Verificar que tengan la columna 'label'
print("Columnas en train:", train_df.columns.tolist())

# Definir las características (todas excepto 'label')
FEATURE_COLS = [col for col in train_df.columns if col != "label"]

# Convertir a arrays de NumPy
X_train = train_df[FEATURE_COLS].values
y_train = train_df["label"].values

X_val = val_df[FEATURE_COLS].values
y_val = val_df["label"].values

X_test = test_df[FEATURE_COLS].values
y_test = test_df["label"].values

# Verificar dimensiones
print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Val:   {X_val.shape}, {y_val.shape}")
print(f"Test:  {X_test.shape}, {y_test.shape}")
print(f"Balance en train: positivo = {y_train.mean():.3f}")

# guardar los arrays en formato .npy para cargarlos después sin pandas
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_val.npy", X_val)
np.save("y_val.npy", y_val)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

print("Archivos .npy guardados correctamente.")

Device: cuda
Columnas en train: ['url_punct_char_count', 'tag_count', 'whitespace_ratio', 'entropy', 'form_count', 'embedded_js_count', 'html_whitespace_ratio', 'script_entropy', 'min_link_length', 'external_link_count', 'total_script_characters', 'internal_link_count', 'url_digit_count', 'label']
Train: (11997, 13), (11997,)
Val:   (4000, 13), (4000,)
Test:  (4000, 13), (4000,)
Balance en train: positivo = 0.500
Archivos .npy guardados correctamente.


In [25]:

# Asegurar que las carpetas existan
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Definición de modelos (adaptados automatomaticamente del json con los parametro de la optimizacion
# =============================================================================

class MLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, output_dim=1, dropout=0.0, activation='relu'):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(prev_dim, n_units))
            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'tanh':
                layers.append(nn.Tanh())
            elif activation == 'leakyrelu':
                layers.append(nn.LeakyReLU(0.1))
            else:
                layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = n_units
        layers.append(nn.Linear(prev_dim, output_dim))
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout, activation):
        super().__init__()
        self.linear1 = nn.Linear(dim, dim)
        self.linear2 = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        if activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        elif activation == 'leakyrelu':
            self.act = nn.LeakyReLU(0.1)
        else:
            self.act = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.act(self.linear1(x))
        out = self.dropout(out)
        out = self.linear2(out)
        out += residual
        out = self.act(out)
        return out


class ResidualMLP(nn.Module):
    def __init__(self, input_dim, hidden_units, n_blocks, output_dim=1, dropout=0.0, activation='relu'):
        super().__init__()
        self.entry = nn.Linear(input_dim, hidden_units)
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_units, dropout, activation) for _ in range(n_blocks)
        ])
        self.exit = nn.Linear(hidden_units, output_dim)
        self.sigmoid = nn.Sigmoid()
        if activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        elif activation == 'leakyrelu':
            self.act = nn.LeakyReLU(0.1)
        else:
            self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.entry(x))
        for block in self.blocks:
            x = block(x)
        x = self.exit(x)
        return self.sigmoid(x)




In [26]:

# =============================================================================
# Funciones auxiliares
# =============================================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device).float().view(-1, 1)
        optimizer.zero_grad()
        outputs = model(Xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, device):
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            outputs = model(Xb).cpu().numpy().ravel()
            preds.extend((outputs >= 0.5).astype(int))
            truths.extend(yb.numpy())
    return np.array(preds), np.array(truths)


def compute_metrics(preds, truths):
    f1 = f1_score(truths, preds, zero_division=0)
    acc = accuracy_score(truths, preds)
    prec = precision_score(truths, preds, zero_division=0)
    cm = confusion_matrix(truths, preds)
    return f1, acc, prec, cm


def measure_inference_time(model, loader, device, num_runs=100):
    model.eval()
    times = []
    with torch.no_grad():
        for Xb, _ in loader:
            Xb = Xb.to(device)
            # warm-up
            for _ in range(10):
                _ = model(Xb)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            for _ in range(num_runs):
                _ = model(Xb)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            end = time.perf_counter()
            total_samples = Xb.size(0) * num_runs
            total_time_us = (end - start) * 1e6
            times.append(total_time_us / total_samples)
    return np.mean(times)


def train_final_model(config, train_dataset, val_dataset, test_dataset,
                      final_epochs, device, models_dir, results_dir):
    arch = config['architecture']
    params = config['params']
    
    # Extraer hiperparámetros comunes
    batch_size = params.get('batch_size', 64)
    lr = params.get('learning_rate', 1e-3)
    weight_decay = params.get('weight_decay', 0.0)
    opt_name = params.get('optimizer', 'adam').lower()
    dropout = params.get('dropout', 0.0)
    activation = params.get('activation', 'relu')
    
    # Construir modelo según arquitectura
    if arch == 'mlp':
        n_layers = params.get('n_layers', 1)
        n_units = params.get('n_units_l0', 128)  # tu JSON usa n_units_l0
        model = MLP(INPUT_DIM, n_layers, n_units, dropout=dropout, activation=activation)
    elif arch == 'residual_mlp':
        # En tu JSON, residual_mlp también tiene n_layers y n_units_l0
        n_blocks = params.get('n_layers', 1)  # número de bloques residuales
        hidden_units = params.get('n_units_l0', 64)
        model = ResidualMLP(INPUT_DIM, hidden_units, n_blocks, dropout=dropout, activation=activation)
    else:
        raise ValueError(f"Arquitectura desconocida: {arch}")
    
    model = model.to(device)
    
    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Criterio y optimizador
    criterion = nn.BCELoss()
    if opt_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif opt_name == 'adamw':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif opt_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Entrenamiento con validación
    best_val_f1 = 0.0
    best_state = None
    training_start = time.time()
    
    for epoch in range(final_epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        preds_val, truths_val = evaluate(model, val_loader, device)
        val_f1, val_acc, val_prec, _ = compute_metrics(preds_val, truths_val)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict().copy()
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:2d}/{final_epochs} | Loss: {train_loss:.4f} | "
                  f"Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")
    
    training_time = time.time() - training_start
    
    # Restaurar mejor modelo
    model.load_state_dict(best_state)
    
    # Evaluación en test
    preds_test, truths_test = evaluate(model, test_loader, device)
    test_f1, test_acc, test_prec, test_cm = compute_metrics(preds_test, truths_test)
    
    # Tiempo de inferencia
    inference_us = measure_inference_time(model, test_loader, device)
    
    # Guardar modelo
    model_filename = f"{arch}_{config['sampler']}_trial{config.get('trial',0)}_f1{test_f1:.4f}.pth"
    model_path = models_dir / model_filename
    torch.save({
        'architecture': arch,
        'params': params,
        'model_state_dict': model.state_dict(),
        'test_f1': test_f1,
        'test_acc': test_acc,
        'test_prec': test_prec,
        'confusion_matrix': test_cm.tolist(),
        'inference_us': inference_us,
        'training_time_s': training_time
    }, model_path)
    
    # Guardar matriz de confusión como imagen
    plt.figure(figsize=(5,4))
    sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - {arch} ({config['sampler']})")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plot_path = results_dir / f"cm_{arch}_{config['sampler']}_trial{config.get('trial',0)}.png"
    plt.savefig(plot_path, dpi=100, bbox_inches='tight')
    plt.close()
    
    # Retornar resumen
    return {
        'architecture': arch,
        'sampler': config['sampler'],
        'trial': config.get('trial'),
        'test_f1': test_f1,
        'test_accuracy': test_acc,
        'test_precision': test_prec,
        'inference_us_per_sample': inference_us,
        'training_time_seconds': training_time,
        'model_path': str(model_path),
        'confusion_matrix_plot': str(plot_path)
    }

In [28]:
# =============================================================================
# Cargar configuración y ejecutar entrenamiento
# =============================================================================

# Cargar JSON (ajusta la ruta si tu archivo está en 'traning/best_config.json')
with open("best_configs.json", "r") as f:
    best_configs = json.load(f)

# Si es un solo objeto, convertirlo a lista
if isinstance(best_configs, dict):
    best_configs = [best_configs]

# Crear datasets de torch (asumiendo que X_train, y_train, etc. ya existen)
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                            torch.tensor(y_val, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                             torch.tensor(y_test, dtype=torch.float32))

# Entrenar cada configuración
all_results = []
for cfg in best_configs:
    print(f"\n===== Entrenando {cfg['architecture']} | sampler={cfg['sampler']} =====")
    res = train_final_model(cfg, train_dataset, val_dataset, test_dataset,
                            final_epochs=FINAL_EPOCHS, device=DEVICE,
                            models_dir=MODELS_DIR, results_dir=RESULTS_DIR)
    all_results.append(res)


===== Entrenando mlp | sampler=tpe =====
Epoch  1/20 | Loss: 0.4508 | Val F1: 0.8344 | Val Acc: 0.8323
Epoch  5/20 | Loss: 0.3691 | Val F1: 0.8597 | Val Acc: 0.8570
Epoch 10/20 | Loss: 0.3481 | Val F1: 0.8659 | Val Acc: 0.8650
Epoch 15/20 | Loss: 0.3465 | Val F1: 0.8695 | Val Acc: 0.8688
Epoch 20/20 | Loss: 0.3391 | Val F1: 0.8778 | Val Acc: 0.8790

===== Entrenando residual_mlp | sampler=nsga2 =====
Epoch  1/20 | Loss: 0.6060 | Val F1: 0.7900 | Val Acc: 0.7708
Epoch  5/20 | Loss: 0.4259 | Val F1: 0.8288 | Val Acc: 0.8200
Epoch 10/20 | Loss: 0.3763 | Val F1: 0.8448 | Val Acc: 0.8448
Epoch 15/20 | Loss: 0.3645 | Val F1: 0.8447 | Val Acc: 0.8455
Epoch 20/20 | Loss: 0.3529 | Val F1: 0.8484 | Val Acc: 0.8482

===== Entrenando mlp | sampler=nsga2 =====
Epoch  1/20 | Loss: 0.4903 | Val F1: 0.8174 | Val Acc: 0.8070
Epoch  5/20 | Loss: 0.4055 | Val F1: 0.8419 | Val Acc: 0.8373
Epoch 10/20 | Loss: 0.3872 | Val F1: 0.8398 | Val Acc: 0.8360
Epoch 15/20 | Loss: 0.3763 | Val F1: 0.8425 | Val Acc: 

In [29]:
# Mostrar resumen
df_results = pd.DataFrame(all_results)
print("\n" + "="*80)
print("RESUMEN FINAL")
print("="*80)
print(df_results[['architecture', 'sampler', 'test_f1', 'test_accuracy',
                  'test_precision', 'inference_us_per_sample', 'training_time_seconds']].to_string(index=False))

# Guardar CSV
df_results.to_csv(RESULTS_DIR / "final_training_metrics.csv", index=False)
print(f"\nResultados guardados en {RESULTS_DIR / 'final_training_metrics.csv'}")


RESUMEN FINAL
architecture sampler  test_f1  test_accuracy  test_precision  inference_us_per_sample  training_time_seconds
         mlp     tpe 0.876500        0.87650        0.876500                 6.612240              19.379167
residual_mlp   nsga2 0.851880        0.85125        0.848290                32.475974              59.924203
         mlp   nsga2 0.859321        0.85700        0.845595                 4.238706              19.652256

Resultados guardados en results\final_training_metrics.csv


In [31]:
df_results = pd.read_csv(RESULTS_DIR / "final_training_metrics.csv")

# Calcular tamaño de cada modelo
sizes_mb = []
for _, row in df_results.iterrows():
    model_path = Path(row['model_path'])
    if model_path.exists():
        size_mb = model_path.stat().st_size / (1024 * 1024)
    else:
        size_mb = None
    sizes_mb.append(size_mb)

df_results['size_mb'] = sizes_mb
print(df_results[['architecture', 'sampler', 'test_f1', 'inference_us_per_sample', 'size_mb']])

   architecture sampler   test_f1  inference_us_per_sample   size_mb
0           mlp     tpe  0.876500                 6.612240  0.016791
1  residual_mlp   nsga2  0.851880                32.475974  0.015082
2           mlp   nsga2  0.859321                 4.238706  0.009660
